# Data Quality

This notebook shows how to detect quality issues in the raw ad tech feeds.

## Create quality and lineage views

This notebook automatically creates the data quality and lineage views before analyzing quality issues.

In [ ]:
%sql
CREATE OR REPLACE VIEW adtech_demo.adtech_platform.vw_data_quality_issues AS
SELECT
  'impressions' AS source_table,
  impression_id AS record_id,
  CASE
    WHEN campaign_id IS NULL THEN 'missing_campaign_id'
    WHEN event_timestamp IS NULL THEN 'missing_timestamp'
    WHEN view_duration_seconds < 0 THEN 'negative_view_duration'
    ELSE NULL
  END AS issue_type,
  campaign_id,
  audience_segment,
  event_timestamp AS event_timestamp,
  view_duration_seconds AS amount,
  publisher AS detail_1,
  placement AS detail_2,
  NULL AS detail_3
FROM adtech_demo.adtech_platform.impressions
WHERE campaign_id IS NULL
   OR event_timestamp IS NULL
   OR view_duration_seconds < 0
UNION ALL
SELECT
  'clicks' AS source_table,
  click_id AS record_id,
  CASE
    WHEN campaign_id IS NULL THEN 'missing_campaign_id'
    WHEN click_timestamp IS NULL THEN 'missing_timestamp'
    WHEN cost < 0 THEN 'negative_cost'
    ELSE NULL
  END AS issue_type,
  campaign_id,
  NULL AS audience_segment,
  click_timestamp AS event_timestamp,
  cost AS amount,
  device AS detail_1,
  geo AS detail_2,
  NULL AS detail_3
FROM adtech_demo.adtech_platform.clicks
WHERE campaign_id IS NULL
   OR click_timestamp IS NULL
   OR cost < 0
UNION ALL
SELECT
  'conversions' AS source_table,
  conversion_id AS record_id,
  CASE
    WHEN campaign_id IS NULL THEN 'missing_campaign_id'
    WHEN conversion_timestamp IS NULL THEN 'missing_timestamp'
    WHEN revenue < 0 THEN 'negative_revenue'
    ELSE NULL
  END AS issue_type,
  campaign_id,
  NULL AS audience_segment,
  conversion_timestamp AS event_timestamp,
  revenue AS amount,
  conversion_type AS detail_1,
  NULL AS detail_2,
  NULL AS detail_3
FROM adtech_demo.adtech_platform.conversions
WHERE campaign_id IS NULL
   OR conversion_timestamp IS NULL
   OR revenue < 0;

CREATE OR REPLACE VIEW adtech_demo.adtech_platform.vw_lineage_summary AS
SELECT
  'campaigns' AS source_table,
  'campaign_id' AS source_column,
  'adtech_demo.adtech_platform.vw_campaign_metrics.campaign_id' AS derived_column,
  'joins impressions, clicks, conversions' AS logic_description
UNION ALL
SELECT
  'impressions' AS source_table,
  'view_duration_seconds' AS source_column,
  'adtech_demo.adtech_platform.vw_campaign_metrics.avg_view_duration_seconds' AS derived_column,
  'average of impression durations' AS logic_description
UNION ALL
SELECT
  'impressions' AS source_table,
  'viewability' AS source_column,
  'adtech_demo.adtech_platform.vw_campaign_metrics.viewability_rate' AS derived_column,
  'ratio of viewable impressions to total impressions' AS logic_description
UNION ALL
SELECT
  'clicks' AS source_table,
  'click_id' AS source_column,
  'adtech_demo.adtech_platform.vw_campaign_metrics.total_clicks' AS derived_column,
  'count of clicks linked to impressions' AS logic_description
UNION ALL
SELECT
  'costs' AS source_table,
  'daily_cost' AS source_column,
  'adtech_demo.adtech_platform.vw_campaign_metrics.total_spend' AS derived_column,
  'sum of campaign daily cost' AS logic_description

In [ ]:
%sql
-- Review raw sources for data quality issues
SELECT * FROM adtech_demo.adtech_platform.vw_data_quality_issues LIMIT 20;

## Common quality checks
- missing campaign ids
- invalid timestamps
- negative costs or revenue
- inconsistent audience or engagement values

In [ ]:
%sql
SELECT source_table, issue_type, COUNT(*) AS issue_count
FROM adtech_demo.adtech_platform.vw_data_quality_issues
GROUP BY source_table, issue_type;